# 09 Results: quantity intent: the twelve VLMBias counting sub-topics

**Question.** Does the quantity intent of the SDK choose the right counter from the question and
count correctly, for the twelve VLMBias counting sub-topics?

**Why.** VLMBias edits each counting image so that the true count differs from the familiar
count (a die face with one dot more, a flag with one star fewer). A model that recalls the
familiar object gives the familiar count.

**Approach.** `scripts/run_counting.py` runs `Harness.answer` with the tools and intents of
`.env.paper`, a fixed classifier that chooses `quantity`, and each item's VLMBias prompt. The
selector chooses the counter from the question (text rules, plus one image check for go and
xiangqi boards). Each counter counts in code; the logo counters get regions from SAM 3. The
items: 795, of which 192 development items (used to write the counters) and 603 evaluation items
(counted once, after the code was frozen) (`data/counting-items.parquet`). The gate is 95% of
evaluation items exact.

**Prediction.** The selector chooses the right counter for every item. Six sub-topics pass the
gate; flag stars, dice, tally, car logos and shoe logos do not.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import runs

RECORDED = runs.recorded_dir()
if RECORDED is None:
    print("INTENT_HARNESS_RECORDED_DIR is not set: the comparison with earlier results is skipped.")
else:
    print("INTENT_HARNESS_RECORDED_DIR is set: the comparison with earlier results runs.")

INTENT_HARNESS_RECORDED_DIR is set: the comparison with earlier results runs.


In [2]:
table = runs.read_result("04-counting")
print("items:", len(table))
print("selector chose the item's sub-topic:", int((table["selected"] == table["sub_topic"]).sum()), "of", len(table))

items: 795
selector chose the item's sub-topic: 795 of 795


## 1. Exact counts by sub-topic

In [3]:
rows = []
for sub_topic, part in table.groupby("sub_topic"):
    development = part[part["split"] == "development"]
    evaluation = part[part["split"] == "evaluation"]
    row = {
        "sub-topic": sub_topic,
        "development": f"{int(development['correct'].sum())} / {len(development)}",
        "evaluation": f"{int(evaluation['correct'].sum())} / {len(evaluation)}" if len(evaluation) else "none",
    }
    if len(evaluation):
        accuracy = evaluation["correct"].mean()
        row["evaluation accuracy"] = round(accuracy, 3)
        row["gate (0.95)"] = "pass" if accuracy >= 0.95 else "fail"
    else:
        row["evaluation accuracy"] = None
        row["gate (0.95)"] = "development only"
    rows.append(row)
by_sub_topic = pd.DataFrame(rows)
evaluation = table[table["split"] == "evaluation"]
print("evaluation items exact:", int(evaluation["correct"].sum()), "of", len(evaluation))
by_sub_topic

evaluation items exact: 453 of 603


,sub-topic,development,evaluation,evaluation accuracy,gate (0.95)
0,car_logos,45 / 63,93 / 117,0.795,fail
1,chess_grid,13 / 13,12 / 12,1.000,pass
2,chess_pieces,13 / 13,60 / 60,1.000,pass
3,dice,6 / 6,45 / 78,0.577,fail
4,flag_stars,18 / 19,61 / 72,0.847,fail
5,flag_stripes,12 / 12,36 / 36,1.000,pass
6,go_grid,13 / 13,none,NaN,development only
7,shoe_logos,1 / 8,17 / 66,0.258,fail
8,sudoku_grid,13 / 13,12 / 12,1.000,pass
9,tally,6 / 6,45 / 78,0.577,fail


## 2. Wrong counts and the familiar count

For each sub-topic: how many wrong evaluation counts equal the familiar count, the count that a
model which does not look would give.

In [4]:
wrong = evaluation[~evaluation["correct"]].copy()
wrong["familiar"] = wrong["count"] == wrong["familiar_count"]
wrong["not found"] = wrong["count"].isna()
wrong.groupby("sub_topic")[["familiar", "not found"]].sum().astype(int).assign(wrong=wrong.groupby("sub_topic").size())

,familiar,not found,wrong
sub_topic,,,
car_logos,3,2,24
dice,33,0,33
flag_stars,3,4,11
shoe_logos,13,0,49
tally,0,0,33


## 3. Comparison with the earlier results, item by item

Runs only when `INTENT_HARNESS_RECORDED_DIR` is set: the count of each item against the earlier
result files (an earlier -1 means "not found", None here).

In [5]:
if RECORDED is not None:
    earlier = []
    for path in sorted(RECORDED.glob("2[3-7]-*.parquet")):
        earlier.append(pd.read_parquet(path)[["item_id", "count"]])
    earlier = pd.concat(earlier)
    earlier["count"] = earlier["count"].where(earlier["count"] != -1)
    both = table.merge(earlier, on="item_id", suffixes=("", "_earlier"))
    same = (both["count"] == both["count_earlier"]) | (both["count"].isna() & both["count_earlier"].isna())
    print("items compared:", len(both), "| same count:", int(same.sum()))
    if (~same).any():
        display(both[~same][["item_id", "sub_topic", "gold_count", "count", "count_earlier"]])

items compared: 795 | same count: 795


## Reading the result

The selector chooses the right counter for all 795 items.

On the 603 evaluation items, 453 counts are exact. Six sub-topics pass the gate of 95%: the three
grids with evaluation items (chess, sudoku, xiangqi), chess pieces, xiangqi pieces and flag
stripes, all at 1.000. Go grid has development items only. Five sub-topics fail: car logos
(0.795), flag stars (0.847), dice (0.577), tally (0.577) and shoe logos (0.258). The prediction is
true.

The wrong counts do not all come from the same cause. For dice, 33 of 33 wrong counts equal the
familiar count. For tally, none of the 33 wrong counts equal the familiar count. For shoe logos,
13 of 49 do. So a wrong count from code is not the same as the bias of a model that does not look.

The comparison with the earlier results gives the same count on 795 of 795 items.

## Conclusion

The quantity intent of the SDK chooses the right counter for every item and counts 453 of 603
evaluation items exactly (0.751). Six of the twelve sub-topics pass the gate. The earlier results
are reproduced exactly.